In [ ]:
import numpy as np
import pickle
from scipy.optimize import fsolve
from scipy.interpolate import interp1d, RegularGridInterpolator

# ============================================================================
# PARTIE 1 : Construction de la lookup table (à faire UNE SEULE FOIS)
# ============================================================================

def build_power_model(power=200, mass_total=70):
    """Construit le modèle puissance → vitesse pour un power donné"""
    g = 9.81
    Crr = 0.004
    rho = 1.225
    CdA = 0.32
    
    def power_equation(v, grade):
        P_gravity = mass_total * g * v * (grade / 100)
        P_rolling = Crr * mass_total * g * v
        P_air = 0.5 * rho * CdA * v**3
        return P_gravity + P_rolling + P_air - power
    
    grades = np.linspace(-10, 20, 301)
    velocities = []
    
    for grade in grades:
        v0 = 15 if grade < 0 else (10 if grade < 5 else 5)
        try:
            v_solution = fsolve(power_equation, v0, args=(grade,))[0]
            velocities.append(v_solution if v_solution > 0 else np.nan)
        except:
            velocities.append(np.nan)
    
    velocities = np.array(velocities)
    valid_mask = ~np.isnan(velocities)
    interpolator = interp1d(grades[valid_mask], velocities[valid_mask], 
                           kind='cubic', bounds_error=False, 
                           fill_value='extrapolate')
    return interpolator


def max_power_time_curve(distance_km):
    """Puissance maximale soutenable selon la distance"""
    if distance_km <= 1:
        return 800
    elif distance_km <= 5:
        return 400
    elif distance_km <= 20:
        return 300
    else:
        return 250


def fatigue_factor_distance(distance_km):
    """Facteur de fatigue selon la distance"""
    if distance_km <= 1:
        return 0.0
    elif distance_km <= 5:
        return 0.2
    elif distance_km <= 10:
        return 0.5
    else:
        return 0.8


def build_lookup_table_3d(
    distance_range=(0.1, 50.0),      # km (min, max)
    distance_step=0.1,                # tous les 100m
    grade_range=(-10, 20),            # % (min, max)
    grade_step=0.5,                   # tous les 0.5%
    segment_distance_range=(0.5, 100),# km totale du segment
    segment_distance_step=0.5,        # tous les 500m
    score_type='physiological'
):
    """
    Construit la lookup table 3D complète.
    
    Dimensions:
    - section_distance: distance de la section (km)
    - section_grade: grade de la section (%)
    - segment_distance: distance totale du segment (km)
    
    Returns:
        dict avec:
            - 'table': array 3D des scores
            - 'axes': dict des axes (distance, grade, segment_distance)
            - 'interpolator': fonction d'interpolation
    """
    
    # Créer les axes
    section_distances = np.arange(distance_range[0], distance_range[1] + distance_step, distance_step)
    section_grades = np.arange(grade_range[0], grade_range[1] + grade_step, grade_step)
    segment_distances = np.arange(segment_distance_range[0], 
                                  segment_distance_range[1] + segment_distance_step, 
                                  segment_distance_step)
    
    print(f"Building lookup table...")
    print(f"  Section distances: {len(section_distances)} points ({distance_range[0]}-{distance_range[1]} km)")
    print(f"  Section grades: {len(section_grades)} points ({grade_range[0]}-{grade_range[1]} %)")
    print(f"  Segment distances: {len(segment_distances)} points ({segment_distance_range[0]}-{segment_distance_range[1]} km)")
    print(f"  Total table size: {len(section_distances) * len(section_grades) * len(segment_distances):,} entries")
    
    # Initialiser la table 3D
    lookup_table = np.zeros((len(section_distances), len(section_grades), len(segment_distances)))
    
    # Pré-calculer les modèles de vitesse pour différentes puissances
    power_models = {}
    unique_powers = np.unique([max_power_time_curve(d) for d in segment_distances])
    for power in unique_powers:
        power_models[power] = build_power_model(power=power, mass_total=88)
    
    # Remplir la table
    total_iterations = len(segment_distances)
    for k, seg_dist in enumerate(segment_distances):
        if k % 20 == 0:
            print(f"  Progress: {k}/{total_iterations} ({k/total_iterations*100:.1f}%)")
        
        max_power = max_power_time_curve(seg_dist)
        fatigue_factor = fatigue_factor_distance(seg_dist)
        velocity_model = power_models[max_power]
        
        for i, sect_dist in enumerate(section_distances):
            for j, grade in enumerate(section_grades):
                # Calculer la vitesse
                velocity = velocity_model(grade)  # m/s
                
                # Temps pour parcourir la section
                time_section = (sect_dist * 1000) / velocity  # secondes
                
                # Position relative (supposée au milieu de la section pour simplifier)
                # Dans la vraie utilisation, on ajustera avec la position réelle
                position_ratio = 0.5  # milieu par défaut
                fatigue_multiplier = 1 + fatigue_factor * position_ratio
                
                # Score selon le type
                if score_type == 'time':
                    score = time_section * fatigue_multiplier
                elif score_type == 'physiological':
                    difficulty_factor = (1 + max(0, grade) / 10) ** 2
                    score = time_section * difficulty_factor * fatigue_multiplier
                else:  # 'energy'
                    energy_kj = (max_power * time_section) / 1000
                    score = energy_kj * fatigue_multiplier
                
                lookup_table[i, j, k] = score
    
    print("✓ Lookup table built successfully!")
    
    # Créer l'interpolateur
    interpolator = RegularGridInterpolator(
        (section_distances, section_grades, segment_distances),
        lookup_table,
        method='linear',
        bounds_error=False,
        fill_value=None
    )
    
    result = {
        'table': lookup_table,
        'axes': {
            'section_distances': section_distances,
            'section_grades': section_grades,
            'segment_distances': segment_distances
        },
        'interpolator': interpolator,
        'score_type': score_type
    }
    
    return result


def save_lookup_table(lookup_dict, filename='physics_lookup_table.pkl'):
    """Sauvegarde la lookup table sur disque"""
    with open(filename, 'wb') as f:
        pickle.dump(lookup_dict, f)
    print(f"✓ Lookup table saved to {filename}")


def load_lookup_table(filename='physics_lookup_table.pkl'):
    """Charge la lookup table depuis le disque"""
    with open(filename, 'rb') as f:
        lookup_dict = pickle.load(f)
    print(f"✓ Lookup table loaded from {filename}")
    return lookup_dict


# ============================================================================
# PARTIE 2 : Utilisation rapide de la lookup table
# ============================================================================

def compute_physics_score_fast(sections, segment_distance_km, lookup_dict):
    """
    Calcule le score physique RAPIDEMENT en utilisant la lookup table.
    
    Args:
        sections: Liste de dict avec 'distance' (m) et 'grade' (%)
        segment_distance_km: Distance totale du segment en km
        lookup_dict: Dictionnaire retourné par build_lookup_table_3d()
    
    Returns:
        float: Score total de difficulté
    """
    interpolator = lookup_dict['interpolator']
    fatigue_factor = fatigue_factor_distance(segment_distance_km)
    
    total_score = 0
    cumulative_distance = 0
    total_distance = sum(s['distance'] for s in sections)
    
    for section in sections:
        sect_dist_km = section['distance'] / 1000  # convertir en km
        grade = section['grade']
        
        # Position relative pour ajuster la fatigue
        position_ratio = cumulative_distance / total_distance if total_distance > 0 else 0.5
        
        # Lookup dans la table (avec position à 0.5 par défaut dans la table)
        base_score = interpolator([sect_dist_km, grade, segment_distance_km])[0]
        
        # Ajuster pour la vraie position
        # Le score de base utilise position_ratio=0.5, on réajuste
        base_fatigue_mult = 1 + fatigue_factor * 0.5
        real_fatigue_mult = 1 + fatigue_factor * position_ratio
        adjusted_score = base_score * (real_fatigue_mult / base_fatigue_mult)
        
        total_score += adjusted_score
        cumulative_distance += section['distance']
    
    return total_score





if __name__ == "__main__":
    lookup_dict = build_lookup_table_3d(
        distance_range=(0.1, 50.0),
        distance_step=0.1,
        grade_range=(-10, 20),
        grade_step=0.5,
        segment_distance_range=(0.5, 100),
        segment_distance_step=0.5,
        score_type='physiological'
    )
    
    # Sauvegarder
    save_lookup_table(lookup_dict, '../src/models/best_rider_physiological_lookup_table.pkl')

    lookup_dict = build_lookup_table_3d(
        distance_range=(0.1, 50.0),
        distance_step=0.1,
        grade_range=(-10, 20),
        grade_step=0.5,
        segment_distance_range=(0.5, 100),
        segment_distance_step=0.5,
        score_type='time'
    )
    
    # Sauvegarder
    save_lookup_table(lookup_dict, '../src/models/best_rider_time_lookup_table.pkl')
    lookup_dict = build_lookup_table_3d(
        distance_range=(0.1, 50.0),
        distance_step=0.1,
        grade_range=(-10, 20),
        grade_step=0.5,
        segment_distance_range=(0.5, 100),
        segment_distance_step=0.5,
        score_type='energy'
    )
    
    # Sauvegarder
    save_lookup_table(lookup_dict, '../src/models/best_rider_energy_lookup_table.pkl')



Building lookup table...
  Section distances: 500 points (0.1-50.0 km)
  Section grades: 61 points (-10-20 %)
  Segment distances: 200 points (0.5-100 km)
  Total table size: 6,100,000 entries
  Progress: 0/200 (0.0%)
  Progress: 20/200 (10.0%)
  Progress: 40/200 (20.0%)
  Progress: 60/200 (30.0%)
  Progress: 80/200 (40.0%)
  Progress: 100/200 (50.0%)
  Progress: 120/200 (60.0%)
  Progress: 140/200 (70.0%)
  Progress: 160/200 (80.0%)
  Progress: 180/200 (90.0%)
✓ Lookup table built successfully!
✓ Lookup table saved to ../src/models/Best_rider_physiological_lookup_table.pkl
Building lookup table...
  Section distances: 500 points (0.1-50.0 km)
  Section grades: 61 points (-10-20 %)
  Segment distances: 200 points (0.5-100 km)
  Total table size: 6,100,000 entries
  Progress: 0/200 (0.0%)
  Progress: 20/200 (10.0%)
  Progress: 40/200 (20.0%)
  Progress: 60/200 (30.0%)
  Progress: 80/200 (40.0%)
  Progress: 100/200 (50.0%)
  Progress: 120/200 (60.0%)
  Progress: 140/200 (70.0%)
  Progres